# 03 — Segmentation Training

Trains one or all 5 folds of a segmentation experiment. Everything about a run — model, loss, optimizer, scheduler, augmentation — is specified in the `EXPERIMENT` dict in Cell 3. Fold control is in Cell 5 (`FOLD_TO_RUN` / `RUN_ALL_FOLDS`).

To switch experiments: edit `RECIPE`, `DATASET`, or `SPLIT_SCHEME` in Cell 3.

**Outputs per fold** (synced to Drive by Cell 11):

```
outputs/
├── checkpoints/<task>/<dataset>/<exp>/fold_k/
│   ├── best.ckpt                ← Lightning checkpoint
│   ├── best_model.pt            ← plain PyTorch state dict + metadata
│   └── experiment_config.json
├── logs/<task>/<dataset>/<exp>/fold_k/metrics.csv
├── tables/<task>/<dataset>/<exp>/<exp>_train_summary.csv
└── figures/<task>/<dataset>/<exp>/
    ├── fold_k/history_fold_k.png
    └── training_curves_overlay.png
```

**Runtime:** GPU runtime (T4 or better). One fold of `01_dice_image_level` takes approximately 30–60 minutes.

## Cell 1 — Install dependencies

In [ ]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

## Cell 2 — Bootstrap: Drive + repo + local SSD scratch

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

In [ ]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

## Cell 3 — EXPERIMENT config (the single source of truth)

Edit `RECIPE`, `DATASET`, and `SPLIT_SCHEME` to define a run. Hyperparameters for each recipe are in `configs/seg/reference_experiments.py`.

**Knobs:**
- `RECIPE`: one of the 7 §13 reference recipes (see `REFERENCE_RECIPES` printed below)
- `DATASET`: `"figshare"` or `"brats2020"`
- `SPLIT_SCHEME`: `"image_level"` (§13 image-level KFold) or `"patient_level"` (no patient leakage)

Fold control is in Cell 5. Dataset-specific throughput overrides (batch_size, lr for brats2020) are applied automatically.

In [ ]:
from configs.seg.reference_experiments import get_experiment, REFERENCE_RECIPES

# ----- The 4 axes of a run -----
RECIPE       = "07_unetpp_effb4_dicebce"        # one of REFERENCE_RECIPES
DATASET      = "figshare"          # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"        # "image_level"  = §13 reference reproduction (image-level KFold)
                                   # "patient_level" = methodologically correct (no patient leakage)
# Fold is set by Cell 5 (FOLD_TO_RUN / RUN_ALL_FOLDS); this value is the registry default.

EXPERIMENT = get_experiment(
    RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1,
)

# ----- Dataset-specific throughput overrides --------------------------------
# figshare: batch_size and lr match the §13 reference recipe exactly.
# brats2020: batch_size 16 with lr scaled by sqrt(16/8) for the linear-scaling rule.
if EXPERIMENT["dataset"] == "figshare":
    EXPERIMENT["num_workers"] = 6
elif EXPERIMENT["dataset"] == "brats2020":
    EXPERIMENT["batch_size"]             = 16
    EXPERIMENT["num_workers"]            = 6
    EXPERIMENT["optimizer_kwargs"]["lr"] = 1.4e-4    # sqrt(16/8) * 1e-4

# ----- Sanity asserts ---------------------------------------------------------
assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2020")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")

print(f"EXPERIMENT: {EXPERIMENT['name']}")
print(f"  recipe:       {EXPERIMENT['recipe']}")
print(f"  dataset:      {EXPERIMENT['dataset']}")
print(f"  split_scheme: {EXPERIMENT['split_scheme']}")
print(f"  model:        {EXPERIMENT['model_name']}")
print(f"  loss:         {EXPERIMENT['loss_name']}  kwargs={EXPERIMENT['loss_kwargs']}")
print(f"  prepro:       {EXPERIMENT['preprocessing']}  aug={EXPERIMENT['augmentation_strength']}")
print(f"  batch_size:   {EXPERIMENT['batch_size']}    num_workers: {EXPERIMENT['num_workers']}")
print(f"  lr:           {EXPERIMENT['optimizer_kwargs']['lr']}")
print(f"\nAvailable recipes: {REFERENCE_RECIPES}")

## Cell 4 — Seed + path resolution

`copy_to_local` auto-detects a zip at `DRIVE_ROOT/data/<dataset>.zip` and
uses it when present: copies one file then extracts locally (~2–3 min).
Falls back to `shutil.copytree` if no zip exists (~60 min for brats2020).
**Create the zip once** by running `01_data_preparation_<dataset>.ipynb` Cell 13.

In [ ]:
from src.train_utils    import set_global_seed
from src.notebook_setup import copy_to_local
from src.file_utils     import experiment_paths, fold_split_csv_paths

set_global_seed(EXPERIMENT["seed"])
LOCAL_ROOT   = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT["dataset"]])
PROJECT_ROOT = LOCAL_ROOT
print(f"LOCAL_ROOT: {LOCAL_ROOT}")


## Cell 5 — Fold Control

In [ ]:
# ----- Fold control --------------------------------------------------------
FOLD_TO_RUN   = 1         # used when RUN_ALL_FOLDS = False
RUN_ALL_FOLDS = True     # flip to True after fold 1 looks good
# ---------------------------------------------------------------------------

folds_to_run = list(range(1, 6)) if RUN_ALL_FOLDS else [FOLD_TO_RUN]
print(f"folds_to_run = {folds_to_run}")
print(f"(experiment: {EXPERIMENT['name']} on {EXPERIMENT['dataset']})")

## Cell 6 — train_one_fold

In [ ]:
import time
import torch
from pytorch_lightning.callbacks import ModelCheckpoint

from src.data_utils         import load_metadata, validate_metadata
from src.sg_data_utils      import build_dataloaders, build_test_loader
from src.sg_models          import build_model, count_parameters
from src.sg_losses          import get_loss
from src.sg_lightning_module import BrainTumorSegModule
from src.train_utils        import (
    build_callbacks, build_trainer, TrainingTimingCallback,
    gather_repro_metadata, export_plain_state_dict, consolidate_metrics_csv,
)
from src.file_utils         import save_json


def train_one_fold(fold: int, experiment: dict) -> dict:
    """
    Train a single fold end-to-end.

    Reads fold CSVs from
        data/<dataset>/splits/<split_scheme>/folds/fold_<k>_{train,val}.csv

    Writes:
        outputs/checkpoints/<task>/<dataset>/<exp>/fold_<k>/best.ckpt
                                                          best_model.pt
                                                          experiment_config.json
        outputs/logs/<task>/<dataset>/<exp>/fold_<k>/metrics.csv
    """
    experiment["fold"] = fold

    # Resolve paths for this fold
    paths = experiment_paths(
        LOCAL_ROOT,
        task=experiment["task"], dataset=experiment["dataset"],
        experiment_name=experiment["name"], fold=fold,
    )
    csv_paths = fold_split_csv_paths(
        LOCAL_ROOT,
        dataset=experiment["dataset"],
        scheme=experiment["split_scheme"],
        fold=fold,
    )

    # ----- Data -----
    train_df = load_metadata(csv_paths["train"]); validate_metadata(train_df)
    val_df   = load_metadata(csv_paths["val"]);   validate_metadata(val_df)
    print(f"  fold {fold}: train={len(train_df):>5} imgs / "
          f"{train_df['patient_id'].nunique():>3} pts | "
          f"val={len(val_df):>5} imgs / {val_df['patient_id'].nunique():>3} pts")

    train_loader, val_loader = build_dataloaders(
        train_df=train_df, val_df=val_df, project_root=LOCAL_ROOT,
        batch_size=experiment["batch_size"],
        num_workers=experiment["num_workers"],
        image_size=experiment["image_size"],
        preprocessing=experiment["preprocessing"],
        augmentation_strength=experiment["augmentation_strength"],
        seed=experiment["seed"],
    )

    # ----- Model + loss + module -----
    model = build_model(
        experiment["model_name"],
        encoder_weights=experiment["encoder_weights"],
    )
    params_count = count_parameters(model)
    loss_fn = get_loss(experiment["loss_name"], **experiment.get("loss_kwargs", {}))

    pl_module = BrainTumorSegModule(
        model=model, loss_fn=loss_fn, threshold=experiment["threshold"],
        optimizer_name=experiment["optimizer_name"],
        optimizer_kwargs=experiment["optimizer_kwargs"],
        scheduler_name=experiment["scheduler_name"],
        scheduler_kwargs=experiment["scheduler_kwargs"],
        scheduler_monitor=experiment["scheduler_monitor"],
        metric_kind=experiment["metric_kind"],
    )

    # ----- Trainer + callbacks -----
    callbacks = build_callbacks(
        ckpt_dir=paths["checkpoints"], monitor="val_dice",
        mode="max", patience=experiment["patience"],
        epoch_summary_keys=("train_loss", "val_loss", "val_dice", "val_iou"),
    )
    trainer = build_trainer(
        callbacks=callbacks, log_dir=paths["logs"],
        max_epochs=experiment["max_epochs"], precision="auto",
    )

    # ----- Train -----
    print(f"  fold {fold}: training start")
    t0 = time.time()
    trainer.fit(pl_module, train_loader, val_loader)
    train_seconds = time.time() - t0
    print(f"  fold {fold}: training done in {train_seconds/60:.1f} min")

    # ----- Collect training meta -----
    ckpt_cb   = next(c for c in callbacks if isinstance(c, ModelCheckpoint))
    timing_cb = next(c for c in callbacks if isinstance(c, TrainingTimingCallback))
    best_ckpt = ckpt_cb.best_model_path
    best_blob = torch.load(best_ckpt, map_location="cpu", weights_only=False)

    # ----- Test evaluation on the best checkpoint -----
    test_df = load_metadata(csv_paths["test"])
    validate_metadata(test_df)
    print(f"  fold {fold}: test evaluation on {len(test_df)} images")
    test_loader = build_test_loader(
        test_df, project_root=LOCAL_ROOT,
        batch_size=experiment["batch_size"],
        num_workers=experiment["num_workers"],
        image_size=experiment["image_size"],
        preprocessing=experiment["preprocessing"],
        return_meta=False,
    )
    test_results = trainer.test(pl_module, test_loader, ckpt_path=best_ckpt, verbose=False)
    test_dice = float(test_results[0]["test_dice"]) if test_results else None
    test_iou  = float(test_results[0]["test_iou"])  if test_results else None
    if test_dice is not None:
        print(f"  fold {fold}: test_dice={test_dice:.4f}  test_iou={test_iou:.4f}")

    training_meta = {
        "fold":                 fold,
        "best_epoch":           int(best_blob.get("epoch", -1)),
        "best_val_dice":        float(ckpt_cb.best_model_score) if ckpt_cb.best_model_score is not None else None,
        "test_dice":            test_dice,
        "test_iou":             test_iou,
        "total_epochs_trained": int(trainer.current_epoch + 1),
        "train_seconds":        train_seconds,
        "params_count":         int(params_count),
        "peak_gpu_mem_mb":      timing_cb.peak_gpu_mem_mb,
        "best_ckpt_path":       best_ckpt,
    }

    # ----- Export plain .pt -----
    export_plain_state_dict(
        lightning_ckpt_path=best_ckpt,
        out_pt_path=paths["best_model"],
        extra_meta={
            "experiment":      experiment,
            "model_name":      experiment["model_name"],
            "encoder_weights": experiment["encoder_weights"],
        },
    )

    # ----- Save experiment_config.json -----
    save_json(
        {
            "EXPERIMENT":     dict(experiment),
            "training_meta":  training_meta,
            "repro_metadata": gather_repro_metadata(repo_root=REPO_ROOT),
        },
        paths["experiment_config_json"],
    )

    # ----- Consolidate + read history CSV for plotting -----
    import pandas as pd
    metrics_csv = paths["logs"] / "metrics.csv"
    history_df = consolidate_metrics_csv(metrics_csv) if metrics_csv.exists() else None

    return {
        **training_meta,
        "history":       history_df,
        "paths":         paths,
        "best_model_pt": str(paths["best_model"]),
    }

print("train_one_fold defined.")

## Cell 7 — Training loop

Per-fold try/except keeps the run going on partial failures. `gc.collect()` and `torch.cuda.empty_cache()` between folds prevent memory fragmentation.

In [ ]:
import gc

results = {}
for fold in folds_to_run:
    print(f"\n{'='*70}")
    print(f"  FOLD {fold}  ({EXPERIMENT['name']} on {EXPERIMENT['dataset']})")
    print(f"{'='*70}")
    try:
        results[fold] = train_one_fold(fold, EXPERIMENT)
        bvd = results[fold]['best_val_dice']
        print(f"\n  ✓ fold {fold} complete: best_val_dice = "
              f"{bvd:.4f}" if bvd is not None else f"  ✓ fold {fold} complete (no val_dice recorded)")
    except Exception as e:
        print(f"\n  ✗ fold {fold} FAILED: {type(e).__name__}: {e}")
        print(f"     to retry: set FOLD_TO_RUN={fold}, RUN_ALL_FOLDS=False in Cell 5, re-run from Cell 6")
        import traceback
        traceback.print_exc()
        results[fold] = {"error": str(e), "fold": fold}
    finally:
        # Force GC + CUDA cleanup between folds — combats memory fragmentation
        # and dataloader thread cleanup. Crucial before fold N+1's model loads.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

print(f"\n{'='*70}")
print(f"  RUN COMPLETE — folds attempted: {list(results.keys())}")
print(f"  failed folds: {[k for k, v in results.items() if 'error' in v]}")
print(f"{'='*70}")

## Cell 8 — Per-fold history plots

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

for fold, res in results.items():
    if "error" in res:
        print(f"fold {fold}: skipped (failed during training)")
        continue
    h = res.get("history")
    if h is None or h.empty:
        print(f"fold {fold}: no metrics.csv found.")
        continue

    # metrics.csv is already consolidated (1 row per epoch)
    h_idx = h.set_index("epoch")

    train_curve = h_idx["train_loss"].dropna()
    val_loss    = h_idx["val_loss"].dropna()
    val_dice    = h_idx["val_dice"].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_curve.index, train_curve.values, label="train_loss")
    axes[0].plot(val_loss.index,    val_loss.values,    label="val_loss")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(loc="upper right")
    axes[0].set_title(f"{EXPERIMENT['name']} fold {fold} — losses")
    axes[0].grid(alpha=0.3)

    axes[1].plot(val_dice.index, val_dice.values, label="val_dice", color="green")
    axes[1].axhline(y=val_dice.max(), linestyle="--", alpha=0.4, color="green",
                    label=f"best={val_dice.max():.4f}")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("dice"); axes[1].legend(loc="lower right")
    axes[1].set_title(f"{EXPERIMENT['name']} fold {fold} — val_dice")
    axes[1].grid(alpha=0.3)

    fig.tight_layout()

    paths = res["paths"]
    fig_dir = paths.get("figures") if "figures" in paths else (
        LOCAL_ROOT / "outputs" / "figures" / EXPERIMENT["task"]
        / EXPERIMENT["dataset"] / EXPERIMENT["name"] / f"fold_{fold}"
    )
    Path(fig_dir).mkdir(parents=True, exist_ok=True)
    out_png = Path(fig_dir) / f"history_fold_{fold}.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  saved {out_png}")

## Cell 9 — All-folds training curve overlay

All folds on one figure: train/val loss (left) and val Dice (right).
Dots mark the best epoch per fold.  
Saved as `outputs/figures/<task>/<dataset>/<exp>/training_curves_overlay.png`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

ok_results = {
    f: r for f, r in results.items()
    if "error" not in r and r.get("history") is not None and not r["history"].empty
}

if not ok_results:
    print("No fold history available — run Cell 7 first.")
else:
    n   = len(ok_results)
    pal = [cm.get_cmap("tab10")(i) for i in range(n)]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for (fold, res), color in zip(sorted(ok_results.items()), pal):
        # metrics.csv is already consolidated (1 row per epoch)
        h = res["history"].set_index("epoch")

        train_loss = h["train_loss"].dropna()
        val_loss   = h["val_loss"].dropna()
        val_dice   = h["val_dice"].dropna()

        if not train_loss.empty:
            axes[0].plot(train_loss.index, train_loss.values,
                         color=color, lw=1.2, alpha=0.7, ls="-")
        if not val_loss.empty:
            axes[0].plot(val_loss.index, val_loss.values,
                         color=color, lw=1.5, alpha=0.9, ls="--",
                         label=f"fold {fold}")

        if not val_dice.empty:
            axes[1].plot(val_dice.index, val_dice.values,
                         color=color, lw=1.5, alpha=0.9,
                         label=f"fold {fold}  best={val_dice.max():.4f}")
            best_ep = int(val_dice.idxmax())
            axes[1].scatter([best_ep], [val_dice.max()], color=color, s=55, zorder=5)

    axes[0].set_xlabel("epoch");  axes[0].set_ylabel("loss")
    axes[0].set_title("Loss  (— train  /  -- val)")
    axes[0].legend(fontsize=8);   axes[0].grid(alpha=0.3)

    axes[1].set_xlabel("epoch");  axes[1].set_ylabel("val Dice")
    axes[1].set_title("Val Dice  (• = best epoch)")
    axes[1].legend(fontsize=8, loc="lower right");  axes[1].grid(alpha=0.3)

    fig.suptitle(
        f"{EXPERIMENT['name']}  ·  {EXPERIMENT['dataset']} "
        f"— training curves overlay  ({n} folds)"
    )
    fig.tight_layout()

    fig_dir = (
        LOCAL_ROOT / "outputs" / "figures"
        / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
    )
    Path(fig_dir).mkdir(parents=True, exist_ok=True)
    out_png = fig_dir / "training_curves_overlay.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_png}")

## Cell 10 — Training summary + per-fold Dice bar chart

Aggregates all fold results into `<exp>_train_summary.csv` (one row per fold) and renders
a test_dice / best_val_dice bar chart. Output is written to local SSD and included in the Cell 11 sync.

In [ ]:
import pandas as pd

rows = []
for fold, res in results.items():
    if "error" in res:
        rows.append({
            "experiment":    EXPERIMENT["name"],
            "dataset":       EXPERIMENT["dataset"],
            "split_scheme":  EXPERIMENT["split_scheme"],
            "fold":          fold,
            "status":        "FAILED",
            "error":         res["error"],
        })
        continue
    rows.append({
        "experiment":           EXPERIMENT["name"],
        "dataset":              EXPERIMENT["dataset"],
        "split_scheme":         EXPERIMENT["split_scheme"],
        "fold":                 fold,
        "status":               "OK",
        "best_epoch":           res["best_epoch"],
        "best_val_dice":        res["best_val_dice"],
        "test_dice":            res.get("test_dice"),
        "test_iou":             res.get("test_iou"),
        "total_epochs_trained": res["total_epochs_trained"],
        "train_minutes":        round(res["train_seconds"] / 60.0, 2),
        "params_count":         res["params_count"],
        "peak_gpu_mem_mb":      res["peak_gpu_mem_mb"],
    })

summary_df = pd.DataFrame(rows)

# Write to outputs/tables/<task>/<dataset>/<exp>/<exp>_train_summary.csv
from pathlib import Path
exp_tables_dir = (
    LOCAL_ROOT / "outputs" / "tables"
    / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
)
exp_tables_dir.mkdir(parents=True, exist_ok=True)
out_csv = exp_tables_dir / f"{EXPERIMENT['name']}_train_summary.csv"
summary_df.to_csv(out_csv, index=False)
print(f"saved {out_csv}")

# ---- styled in-notebook display ----
from IPython.display import display

ok_mask = summary_df["status"] == "OK"
ok_df   = summary_df[ok_mask]

fmt = {
    "best_val_dice":   "{:.4f}",
    "test_dice":       "{:.4f}",
    "test_iou":        "{:.4f}",
    "train_minutes":   "{:.1f}",
    "params_count":    "{:,.0f}",
}

def _row_style(row):
    return ["background-color: #ffe0e0" if row["status"] == "FAILED" else "" for _ in row]

if ok_mask.any():
    dice_vals = ok_df["test_dice"].dropna()
    vmin = float(dice_vals.min()) - 0.01 if not dice_vals.empty else 0.0
    vmax = float(dice_vals.max()) + 0.01 if not dice_vals.empty else 1.0
    styled = (
        summary_df.style
        .format(fmt, na_rep="\u2014")
        .background_gradient(subset=["test_dice", "best_val_dice"], cmap="RdYlGn",
                             vmin=vmin, vmax=vmax)
        .apply(_row_style, axis=1)
        .set_caption(
            f"{EXPERIMENT['name']} \u00b7 {EXPERIMENT['dataset']} \u00b7 {EXPERIMENT['split_scheme']}"
        )
    )
else:
    styled = summary_df.style.format(fmt, na_rep="\u2014").apply(_row_style, axis=1)

display(styled)

# ---- cross-fold summary print ----
if ok_mask.any():
    print(f"\n\u2500\u2500 {EXPERIMENT['name']}  ({len(ok_df)}/{len(summary_df)} folds OK) \u2500\u2500")
    for col, lbl in [
        ("test_dice",     "test_dice    "),
        ("test_iou",      "test_iou     "),
        ("best_val_dice", "best_val_dice"),
        ("train_minutes", "train_min    "),
    ]:
        vals = ok_df[col].dropna()
        if vals.empty:
            continue
        s = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        print(f"  {lbl}  {vals.mean():.4f} \u00b1 {s:.4f}"
              f"  [min {vals.min():.4f}  max {vals.max():.4f}]")

# ---- per-fold dice bar chart ----
import matplotlib.pyplot as plt
import numpy as np

if not ok_df.empty:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    x = np.arange(len(ok_df))
    w = 0.35
    ax.bar(x - w/2, ok_df["test_dice"].values,     width=w,
           label="test_dice",     color="#2196F3", alpha=0.85)
    ax.bar(x + w/2, ok_df["best_val_dice"].values, width=w,
           label="best_val_dice", color="#4CAF50", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"fold {f}" for f in ok_df["fold"]])
    ax.set_ylabel("Dice score")
    all_vals = pd.concat([ok_df["test_dice"], ok_df["best_val_dice"]]).dropna()
    ax.set_ylim(max(0.0, float(all_vals.min()) - 0.05), 1.0)
    ax.axhline(float(ok_df["test_dice"].mean()), linestyle="--", color="#2196F3",
               alpha=0.5, label=f"mean test = {ok_df['test_dice'].mean():.4f}")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    ax.set_title(
        f"{EXPERIMENT['name']} \u00b7 {EXPERIMENT['dataset']} \u2014 per-fold dice"
    )
    fig.tight_layout()

    fig_dir = (
        LOCAL_ROOT / "outputs" / "figures"
        / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
    )
    Path(fig_dir).mkdir(parents=True, exist_ok=True)
    out_png = Path(fig_dir) / f"{EXPERIMENT['name']}_fold_dice_bar.png"
    fig.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"  saved {out_png}")


## Cell 11 — Sync to Drive

In [ ]:
from src.notebook_setup import sync_outputs_to_drive

sync_outputs_to_drive(
    drive_root=DRIVE_ROOT, local_root=LOCAL_ROOT,
    task=EXPERIMENT["task"], dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
    categories=["checkpoints", "logs", "tables", "figures"],
)
print("sync complete")

In [ ]:
# Only terminate if sync above actually completed successfully
SYNC_OK = True   # set this manually based on the sync cell's output
if SYNC_OK:
    from google.colab import runtime
    print("Disconnecting runtime in 3 seconds...")
    import time; time.sleep(3)
    runtime.unassign()
else:
    print("SYNC_OK is False — staying connected. Investigate the sync cell first.")

In [ ]:
# ----- Terminate the Colab session -------------------------------------------
# Runs only after sync_outputs_to_drive() above has completed, so everything
# is safely on Drive. After this call:
#   - the runtime disconnects
#   - all in-memory state is lost
#   - compute units stop being consumed
#   - the local SSD copy under /content/Senior_Project_local is wiped
# Reconnecting later starts a fresh runtime (re-run from Cell 1)..

from google.colab import runtime
print("All folds complete. Disconnecting runtime in 3 seconds to save compute...")
import time; time.sleep(3)
runtime.unassign()